In [2]:
from ucimlrepo import fetch_ucirepo 
import pandas as pd  
import numpy as np
 
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_absolute_percentage_error, r2_score, mean_squared_error, mean_absolute_error, accuracy_score
from sklearn.metrics import r2_score
from sklearn.feature_selection import SelectFromModel

In [3]:
# 讀取數據
data = pd.read_csv("Boston.csv")

# 拆分為特徵跟目標
x = data.drop(['medv'], axis=1)  # 特徵
y = data['medv']  # 目標
print(x.shape)
print(y.shape)

(506, 14)
(506,)


In [4]:
train_X, test_X, train_y, test_y = train_test_split(x, y, test_size = 0.3, random_state = 25) 
# train_X, valid_X, train_y, valid_y = train_test_split(train_X, train_y, test_size = 0.3, random_state = 25)
model = XGBRegressor(max_depth = 4)
model.fit(train_X, train_y)
print(model.feature_importances_)



[0.01967136 0.02281407 0.00254272 0.00646287 0.00059794 0.06737426
 0.4652595  0.00813038 0.05468912 0.01654823 0.04790234 0.07115714
 0.00801835 0.20883174]


In [5]:
thresholds = np.sort(model.feature_importances_)
print(len(thresholds))
print (thresholds)

14
[0.00059794 0.00254272 0.00646287 0.00801835 0.00813038 0.01654823
 0.01967136 0.02281407 0.04790234 0.05468912 0.06737426 0.07115714
 0.20883174 0.4652595 ]


In [6]:
r2_scores_vaild = []
MAPEs_vaild = []
MSEs_vaild = []

r2_scores_test = []
MAPEs_test = []
MSEs_test = [] 

max_r2_score = 0
max_MAPE = 0
max_MSE = 0

In [7]:
from sklearn.model_selection import KFold
kf = KFold(n_splits=5, shuffle=True, random_state=0)  # 5 fold交叉驗證

average_r2_v = []
average_mape_v = []
average_rmse_v= []

average_r2_t= []
average_mape_t= []
average_rmse_t= []

for thresh in thresholds:
    r2_scores_v = []
    mape_scores_v = []
    rmse_scores_v = []

    r2_scores_t = []
    mape_scores_t = []
    rmse_scores_t = []

    for fold, (train_index, test_index) in enumerate(kf.split(train_X), 1):
        X_train, X_test = train_X.iloc[train_index], train_X.iloc[test_index]
        y_train, y_test = train_y.iloc[train_index], train_y.iloc[test_index]


        # select features using threshold
        selection = SelectFromModel(model, threshold = thresh, prefit = True)
        select_X_train = selection.transform(X_train)
        # train model
        selection_model = XGBRegressor(n_jobs=-1, max_depth = 3)
        selection_model.fit(select_X_train, y_train)
        #print(model.score(train_X, train_y))
        # eval model
        select_X_vaild = selection.transform(X_test)
        select_X_test = selection.transform(test_X)
        predictions_vaild = selection_model.predict(select_X_vaild)
        predictions_test = selection_model.predict(select_X_test)
        # print("---------------------------------------")
        # print('Thresh=%.3f, n=%d' % (thresh, select_X_train.shape[1]))
        # print("r2_score:", r2_score(y_test, predictions_vaild))
        # print("MAPE:", mean_absolute_percentage_error(y_test, predictions_vaild))
        # print("RMSE:", np.sqrt(mean_squared_error(y_test, predictions_vaild)))
        # print("---------------------------------------")

        # print("---------------------------------------")
        # print('Thresh=%.3f, n=%d' % (thresh, select_X_train.shape[1]))
        # print("r2_score:", r2_score(test_y, predictions_test))
        # print("MAPE:", mean_absolute_percentage_error(test_y, predictions_test))
        # print("RMSE:", np.sqrt(mean_squared_error(test_y, predictions_test)))
        # print("---------------------------------------")
        r2_scores_vaild.append(r2_score(y_test, predictions_vaild))
        MAPEs_vaild.append(mean_absolute_percentage_error(y_test, predictions_vaild))
        MSEs_vaild.append(np.sqrt(mean_squared_error(y_test, predictions_vaild)))

        # r2_scores_test.append(r2_score(test_y, predictions_test))
        # MAPEs_test.append(mean_absolute_percentage_error(test_y, predictions_test))
        # MSEs_test.append(np.sqrt(mean_squared_error(test_y, predictions_test)))

        print("-------------------")
        print("Vaild")
        ############      Vaild
        print('Thresh=%.3f, n=%d' % (thresh, select_X_train.shape[1]))
        y_pred = selection_model.predict(select_X_vaild)

        r2 = r2_score(y_test, y_pred)
        r2_scores_v.append(r2)

        # 計算 MAPE
        mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
        mape_scores_v.append(mape)

        print(f"Fold {fold}: MAPE = {mape:.2f}%")
        #計算 r2
        print(f"Fold {fold}: R-squared = {r2}")

        # 計算 RMSE
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        rmse_scores_v.append(rmse)

        print(f"Fold {fold}: RMSE = {rmse:.2f}")

        print("---------------------")
        print("test")
        ############      test
        print('Thresh=%.3f, n=%d' % (thresh, select_X_train.shape[1]))
        y_pred = selection_model.predict(select_X_test)

        r2 = r2_score(test_y, y_pred)
        r2_scores_t.append(r2)


        # 計算 MAPE
        mape = np.mean(np.abs((test_y - y_pred) / test_y)) * 100
        mape_scores_t.append(mape)

        print(f"Fold {fold}: MAPE = {mape:.2f}%")
        #計算 r2
        print(f"Fold {fold}: R-squared = {r2}")

        # 計算 RMSE
        rmse = np.sqrt(mean_squared_error(test_y, y_pred))
        rmse_scores_t.append(rmse)

        print(f"Fold {fold}: RMSE = {rmse:.2f}")
        
    print("---------------------------------------------------------")
    print("Vaild")
    average_r2_v.append(np.mean(r2_scores_v)) 
    print("Average R-squared across folds:", np.mean(r2_scores_v))

    average_mape_v.append(np.mean(mape_scores_v)) 
    print("Average MAPE across folds:", np.mean(mape_scores_v))

    average_rmse_v.append(np.mean(rmse_scores_v)) 
    print("Average RMSE across folds:", np.mean(rmse_scores_v)) 

    print("---------------------------------------------------------")
    print("test")
    average_r2_t.append(np.mean(r2_scores_t)) 
    print("Average R-squared across folds:", np.mean(r2_scores_t))

    average_mape_t.append(np.mean(mape_scores_t)) 
    print("Average MAPE across folds:", np.mean(mape_scores_t))

    average_rmse_t.append(np.mean(rmse_scores_t)) 
    print("Average RMSE across folds:", np.mean(rmse_scores_t)) 



-------------------
Vaild
Thresh=0.001, n=14
Fold 1: MAPE = 14.18%
Fold 1: R-squared = 0.8941370652309126
Fold 1: RMSE = 3.43
---------------------
test
Thresh=0.001, n=14
Fold 1: MAPE = 11.13%
Fold 1: R-squared = 0.883501663778417
Fold 1: RMSE = 2.96


c:\Users\1\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
c:\Users\1\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
c:\Users\1\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
c:\Users\1\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
c:\Users\1\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
c:\Users\1\AppData\Local\Progr

-------------------
Vaild
Thresh=0.001, n=14
Fold 2: MAPE = 10.05%
Fold 2: R-squared = 0.8676988105015174
Fold 2: RMSE = 3.05
---------------------
test
Thresh=0.001, n=14
Fold 2: MAPE = 9.49%
Fold 2: R-squared = 0.9094920562115392
Fold 2: RMSE = 2.61
-------------------
Vaild
Thresh=0.001, n=14
Fold 3: MAPE = 11.64%
Fold 3: R-squared = 0.8258122945289186
Fold 3: RMSE = 3.59
---------------------
test
Thresh=0.001, n=14
Fold 3: MAPE = 10.51%
Fold 3: R-squared = 0.885549984850493
Fold 3: RMSE = 2.94
-------------------
Vaild
Thresh=0.001, n=14
Fold 4: MAPE = 10.86%
Fold 4: R-squared = 0.8057107600836351
Fold 4: RMSE = 4.76
---------------------
test
Thresh=0.001, n=14
Fold 4: MAPE = 10.77%
Fold 4: R-squared = 0.812826614799373
Fold 4: RMSE = 3.76
-------------------
Vaild
Thresh=0.001, n=14
Fold 5: MAPE = 13.57%
Fold 5: R-squared = 0.7982235250789841
Fold 5: RMSE = 3.64
---------------------
test
Thresh=0.001, n=14
Fold 5: MAPE = 10.20%
Fold 5: R-squared = 0.9032203470760577
Fold 5: RMS

c:\Users\1\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
c:\Users\1\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
c:\Users\1\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
c:\Users\1\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
c:\Users\1\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
c:\Users\1\AppData\Local\Progr

-------------------
Vaild
Thresh=0.003, n=13
Fold 1: MAPE = 14.32%
Fold 1: R-squared = 0.8994388534750459
Fold 1: RMSE = 3.35
---------------------
test
Thresh=0.003, n=13
Fold 1: MAPE = 11.01%
Fold 1: R-squared = 0.8788009113829802
Fold 1: RMSE = 3.02
-------------------
Vaild
Thresh=0.003, n=13
Fold 2: MAPE = 10.05%
Fold 2: R-squared = 0.8676988105015174
Fold 2: RMSE = 3.05
---------------------
test
Thresh=0.003, n=13
Fold 2: MAPE = 9.49%
Fold 2: R-squared = 0.9094920562115392
Fold 2: RMSE = 2.61


...

c:\Users\1\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
c:\Users\1\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
c:\Users\1\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
c:\Users\1\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
c:\Users\1\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
c:\Users\1\AppData\Local\Progr

-------------------
Vaild
Thresh=0.209, n=2
Fold 1: MAPE = 19.98%
Fold 1: R-squared = 0.8195053554357203
Fold 1: RMSE = 4.48
---------------------
test
Thresh=0.209, n=2
Fold 1: MAPE = 17.21%
Fold 1: R-squared = 0.6667357676389778
Fold 1: RMSE = 5.01
-------------------
Vaild
Thresh=0.209, n=2
Fold 2: MAPE = 19.16%
Fold 2: R-squared = 0.6439991004644775
Fold 2: RMSE = 5.00
---------------------
test
Thresh=0.209, n=2
Fold 2: MAPE = 16.86%
Fold 2: R-squared = 0.6704350862442184
Fold 2: RMSE = 4.98
-------------------
Vaild
Thresh=0.209, n=2
Fold 3: MAPE = 15.47%
Fold 3: R-squared = 0.6914020201324556
Fold 3: RMSE = 4.78
---------------------
test
Thresh=0.209, n=2
Fold 3: MAPE = 16.58%
Fold 3: R-squared = 0.7022161017016242
Fold 3: RMSE = 4.74
-------------------
Vaild
Thresh=0.209, n=2
Fold 4: MAPE = 15.31%
Fold 4: R-squared = 0.7243013267397052
Fold 4: RMSE = 5.67
---------------------
test
Thresh=0.209, n=2
Fold 4: MAPE = 16.73%
Fold 4: R-squared = 0.7027554610845556
Fold 4: RMSE = 4

c:\Users\1\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
c:\Users\1\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
c:\Users\1\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
c:\Users\1\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
c:\Users\1\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
c:\Users\1\AppData\Local\Progr

-------------------
Vaild
Thresh=0.209, n=2
Fold 5: MAPE = 20.68%
Fold 5: R-squared = 0.5529991605960463
Fold 5: RMSE = 5.42
---------------------
test
Thresh=0.209, n=2
Fold 5: MAPE = 15.84%
Fold 5: R-squared = 0.7075752236900941
Fold 5: RMSE = 4.69
---------------------------------------------------------
Vaild
Average R-squared across folds: 0.6864413926736811
Average MAPE across folds: 18.11919826960846
Average RMSE across folds: 5.07174865017703
---------------------------------------------------------
test
Average R-squared across folds: 0.689943528071894
Average MAPE across folds: 16.643348731436827
Average RMSE across folds: 4.831450240900521
-------------------
Vaild
Thresh=0.465, n=1
Fold 1: MAPE = 29.94%
Fold 1: R-squared = 0.6785227399527556
Fold 1: RMSE = 5.98
---------------------
test
Thresh=0.465, n=1
Fold 1: MAPE = 27.01%
Fold 1: R-squared = 0.3528145386038323
Fold 1: RMSE = 6.98
-------------------
Vaild
Thresh=0.465, n=1
Fold 2: MAPE = 35.55%
Fold 2: R-squared = -0.0

c:\Users\1\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
c:\Users\1\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
c:\Users\1\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
c:\Users\1\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
c:\Users\1\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(


In [8]:
print("MAX average R-squared : N=",average_r2_t[::-1].index(max(average_r2_t))+1 ,", R-squared=" ,max(average_r2_t))
print("MAX average MAPE : N=" , average_mape_t[::-1].index(min(average_mape_t))+1,", MAPE=" , min(average_mape_t))
print("MAX average RMSE : N=", average_rmse_t[::-1].index(min(average_rmse_t))+1,", RMSE=" , min(average_rmse_t))


MAX average R-squared : N= 11 , R-squared= 0.8801126074058045
MAX average MAPE : N= 11 , MAPE= 10.399603714109443
MAX average RMSE : N= 11 , RMSE= 2.979396548364195
